In [ ]:
import os
import random
import time
import numpy as np
import tensorflow as tf
import tensorflow.keras
import tensorflow.keras.applications
tf.__version__

In [ ]:
IMG_SIZE = 512
# BATCH_SIZE = 4
BATCH_SIZE = 8
# BATCH_SIZE = 16
VAL_SPLIT = 0.05
SEED = 42

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

In [ ]:
X = np.load('X.npy')
Y = np.load('Y.npy').reshape(-1)
# X.shape, X.min(), X.max(), X.mean(), X.std(), X.dtype
# Y.shape, Y.min(), Y.max(), Y.mean(), Y.std(), Y.dtype
# ((63179,), 0.0, 1.0, 0.04460343, 0.2064315, dtype('float32'))

In [ ]:
N = len(Y)

def batch_generator(X, Y, ids, batch_size=BATCH_SIZE, shuffle=True):
    pos_weight = 50.0
    neg_weight = 1.0
    
    N = len(ids)
    while True:
        if shuffle:
            np.random.shuffle(ids)
        for start in range(0, N, batch_size):
            end = min(start + batch_size, N)
            batch = ids[start:end]
            X_ = X[batch]
            Y_ = Y[batch]
            
            #weights = np.where(Y_ == 1, pos_weight, neg_weight)
            yield X_, Y_#, weights
            
            #yield X_, Y_

indices = np.arange(N)
np.random.shuffle(indices)
id_train = indices[:int(N*(1-VAL_SPLIT))]
id_valid = indices[int(N*(1-VAL_SPLIT)):]

gen_train = batch_generator(X, Y, id_train, batch_size=BATCH_SIZE, shuffle=True)
gen_valid = batch_generator(X, Y, id_valid, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
N = len(Y)

def batch_generator_balanced(X, Y, ids, batch_size=BATCH_SIZE, shuffle=True):
    N = len(ids)
    half_batch = batch_size // 2
    pos_idx = ids[Y[ids] == 1]
    neg_idx = ids[Y[ids] == 0]
    while True:
        pos_batch = np.random.choice(pos_idx, half_batch)
        neg_batch = np.random.choice(neg_idx, half_batch)
        batch = np.concatenate([pos_batch, neg_batch])
        np.random.shuffle(batch)
        yield X[batch], Y[batch]

gen_train_balanced = batch_generator_balanced(X, Y, id_train, batch_size=BATCH_SIZE, shuffle=True)

In [ ]:
data_augmentation = tf.keras.Sequential(
    [
        tensorflow.keras.layers.RandomFlip("horizontal"),
        tensorflow.keras.layers.RandomRotation(0.05),
        tensorflow.keras.layers.RandomZoom(0.1),
    ],
    name="augmentation"
)

In [ ]:
BaseModel = tf.keras.applications.ConvNeXtBase
BaseModelPreProcess = tf.keras.applications.convnext.preprocess_input
ModelName = 'ConvNeXtBase'

# BaseModel = tf.keras.applications.ResNet50
# BaseModelPreProcess = tf.keras.applications.resnet.preprocess_input
# ModelName = 'ResNet50'

# BaseModel = tf.keras.applications.ConvNeXtSmall
# BaseModelPreProcess = tf.keras.applications.convnext.preprocess_input
# ModelName = 'ConvNeXtSmall'

# BaseModel = tf.keras.applications.EfficientNetB0
# BaseModelPreProcess = tf.keras.applications.efficientnet.preprocess_input
# ModelName = 'EfficientNetB0'

# BaseModel = tf.keras.applications.EfficientNetB5
# BaseModelPreProcess = tf.keras.applications.efficientnet.preprocess_input

# BaseModel = tf.keras.applications.EfficientNetB6
# BaseModelPreProcess = tf.keras.applications.efficientnet.preprocess_input

# BaseModel = tf.keras.applications.EfficientNetB7
# BaseModelPreProcess = tf.keras.applications.efficientnet.preprocess_input

In [ ]:
inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 1), name="input_image")

# Augmentation (training only)
# x = data_augmentation(inputs) # marche mieux sans
x = inputs

# 1 -> 3 channel adapter
x = tf.keras.layers.Conv2D(
    filters=3,
    kernel_size=1,
    padding="same",
    use_bias=False,
    name="gray_to_rgb"
)(x)

# Preprocessing for ImageNet backbone
x = BaseModelPreProcess(x)

# Backbone
backbone = BaseModel(
    include_top=False,
    weights="imagenet",
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# with or without freezing ?
# backbone.trainable = False
x = backbone(x, training=True)

# Classification head
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.BatchNormalization()(x)

x = tf.keras.layers.Dense(512, activation="relu")(x)
# x = tf.keras.layers.Dropout(0.5)(x)

outputs = tf.keras.layers.Dense(1, activation="sigmoid", name="output")(x)

model = tf.keras.models.Model(inputs, outputs)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(),#optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.BinaryCrossentropy(), # BinaryFocalCrossentropy
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="acc"),
        tf.keras.metrics.AUC(name="auc"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)

In [ ]:
model.summary()

In [ ]:
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    'best_model.keras',
    monitor='val_loss',
    save_best_only=True,
    mode='min',
    verbose=1
)

In [ ]:
history = model.fit(
    gen_train_balanced, #gen_train
    steps_per_epoch=len(id_train)//BATCH_SIZE//2,
    validation_data=gen_valid,
    validation_steps=len(id_valid)//BATCH_SIZE,
    batch_size=BATCH_SIZE,
    epochs=100,
    callbacks=[checkpoint]
)

In [ ]:
model.save(f'{ModelName}_100e_BCE.keras')

In [ ]:
print(f'{ModelName}_100e_BCE.keras')